## Changelog
- parent: 20260506_133936_2e50b541
- change: new baseline — keep only the top-20 mutual-information columns and use
    a minimal preprocessing pipeline (median impute + ordinal encode), no FE.
- hypothesis: the previous pipeline carries 75 columns plus derived features and
    fits a 23-column garage block compressed via PLS. A leaner 20-column input with
    no engineered features sets a clean reference point for measuring how much each
    later step (TotalSF, temporal features, garage-PLS, target encoding) actually
    contributes vs. just 'use the strongest signals'.

In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")

In [ ]:
# top-20 mutual-information features (see eda-mutual-information.ipynb)
TOP_FEATURES = [
    "OverallQual", "Neighborhood", "GarageArea", "GrLivArea", "YearBuilt",
    "TotalBsmtSF", "LotArea", "GarageCars", "ExterQual", "KitchenQual",
    "BsmtQual", "1stFlrSF", "YearRemodAdd", "GarageYrBlt", "MSSubClass",
    "GarageFinish", "FullBath", "LotFrontage", "FireplaceQu", "TotRmsAbvGrd",
]
# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
CATEGORICAL = [
    "Neighborhood", "ExterQual", "KitchenQual", "BsmtQual",
    "MSSubClass", "GarageFinish", "FireplaceQu",
]
NUMERIC = [c for c in TOP_FEATURES if c not in CATEGORICAL]

# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

X = train_data[TOP_FEATURES].copy()
X["MSSubClass"] = X["MSSubClass"].astype(str)
y = np.log1p(train_data["SalePrice"])

X_test = test_data_raw[TOP_FEATURES].copy()
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold

# ColumnTransformer: List of (name, transformer, columns) tuples specifying the transformer objects to be applied to subsets of the data.
# transformer: {‘drop’, ‘passthrough’} or estimator. Estimator must support fit and transform. 
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC),  # Replace missing values using median along each column
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ord",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), CATEGORICAL),
])

pipe = Pipeline([
    ("prep",  preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=300, random_state=42)),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X, y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rmse = -scores
print(f"CV RMSE (log-price): {rmse.mean():.4f} ± {rmse.std():.4f}")
print(f"Per-fold:            {np.round(rmse, 4).tolist()}")

pipe.fit(X, y)

In [ ]:
test_pred = np.expm1(pipe.predict(X_test))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)